# 2. Фурье-пространство и перенос решения на приёмник

Здесь ещё не строим пространственную таблицу. Цель — получить обычную
линейную систему на одном $(k,\omega)$ и понять, почему она не зависит
от направления первоначального фотона.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium
from lighthit.single import hg_phase

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())
# Одна точка преобразования на всю тетрадь.
K_PER_M, OMEGA_PER_NS = 0.35, 0.08


## 2.1. Соглашение о Фурье
$$\widetilde f(\mathbf k,\omega)=\int f(\mathbf x,t)e^{-i\mathbf k\cdot\mathbf x+i\omega t}d^3xdt.$$
Тогда $\partial_t\to-i\omega$, $\nabla\to i\mathbf k$. В однородной среде
разные $\mathbf k$ не связаны:
$$(D-V)\widetilde I=\widetilde q,\quad
D=d_0+ik\mu,\quad d_0=\mu_t-i\omega/v,\quad\mu=\widehat{\mathbf k}\cdot\mathbf s.$$
$V$ — интегральный оператор рассеяния, $(Vf)(\mathbf s)=\mu_s\int p(\mathbf s\cdot\mathbf s')f(\mathbf s')d\Omega'$.

Приёмник измеряет $\widetilde K=\int\widetilde I\,d\Omega$.
С билинейным спариванием $[f,g]=\int fg\,d\Omega$ (без комплексного сопряжения)
оператор $D-V$ симметричен. Для направленной вспышки:
$$\widetilde K=[1,(D-V)^{-1}\delta_{\mathbf s_0}]
=[(D-V)^{-1}1,\delta_{\mathbf s_0}]=h(\mathbf s_0),\qquad(D-V)h=1.$$

Правая часть единица соответствует изотропному приёмнику. Произвольное
направление источника $\mathbf s_0$ подставляется **после** решения. Для
направленного приёмника правой частью была бы его функция чувствительности;
как это устроено эффективно — глава 10, здесь только сам принцип.


In [ ]:
v = medium.speed_m_per_ns
d0 = medium.extinction_per_m - 1j*OMEGA_PER_NS/v
print('k [1/m] =', K_PER_M, ' omega [rad/ns] =', OMEGA_PER_NS, ' d0 [1/m] =', d0)

# 1/D как функция косинуса стриминга: множитель, и ничего больше.
mu_grid = np.linspace(-1, 1, 501)
free = 1/(d0 + 1j*K_PER_M*mu_grid)
fig, ax = plt.subplots()
ax.plot(mu_grid, free.real, label='Re 1/D')
ax.plot(mu_grid, free.imag, label='Im 1/D')
ax.set(xlabel='mu = k-hat . s', ylabel='длина [м]')
ax.legend(); plt.show()


## 2.2. Знаки преобразования и теорема сдвига

Интегрирование по частям даёт $\partial_t\to-i\omega$ и $\nabla\to+i\mathbf k$ —
знаки разные. Сдвиг источника даёт
$$q(\mathbf x-\mathbf x_0,t-t_0)\to e^{-i\mathbf k\cdot\mathbf x_0+i\omega t_0}\widetilde q.$$
Пространственную и временную фазы проверяем **по отдельности**: совместная
проверка прошла бы и при двух перевёрнутых знаках. Производные берём
аналитически, чтобы не примешивать ошибку конечной разности.


In [ ]:
x = np.linspace(-40., 40., 4001)
sigma_x, sigma_t, x0, t0 = 1.3, 0.9, 2.5, 1.75
gauss = lambda grid, width, centre=0.: np.exp(-0.5*((grid-centre)/width)**2)

space, space_shift = gauss(x, sigma_x), gauss(x, sigma_x, x0)
time, time_shift = gauss(x, sigma_t), gauss(x, sigma_t, t0)
space_hat = np.trapezoid(space*np.exp(-1j*K_PER_M*x), x)
time_hat = np.trapezoid(time*np.exp(1j*OMEGA_PER_NS*x), x)

spatial_phase = np.trapezoid(space_shift*np.exp(-1j*K_PER_M*x), x)/space_hat
temporal_phase = np.trapezoid(time_shift*np.exp(1j*OMEGA_PER_NS*x), x)/time_hat
d_space = np.trapezoid((-(x/sigma_x**2)*space)*np.exp(-1j*K_PER_M*x), x)/space_hat
d_time = np.trapezoid((-(x/sigma_t**2)*time)*np.exp(1j*OMEGA_PER_NS*x), x)/time_hat

print('пространственная фаза', spatial_phase, 'ожидание', np.exp(-1j*K_PER_M*x0))
print('временная фаза      ', temporal_phase, 'ожидание', np.exp(1j*OMEGA_PER_NS*t0))
print('d/dx ->', d_space, 'ожидание', 1j*K_PER_M)
print('d/dt ->', d_time, 'ожидание', -1j*OMEGA_PER_NS)
assert abs(spatial_phase - np.exp(-1j*K_PER_M*x0)) < 1e-9
assert abs(temporal_phase - np.exp(1j*OMEGA_PER_NS*t0)) < 1e-9
assert abs(d_space - 1j*K_PER_M) < 1e-9 and abs(d_time + 1j*OMEGA_PER_NS) < 1e-9


## 2.3. Плотная угловая сетка и билинейная симметрия

Строим независимую проверочную дискретизацию: Гаусс–Лежандр по $\cos\theta$
(полярная ось вдоль $\mathbf k$) и равномерная сетка по $\varphi$. На такой
сетке билинейное спаривание — это $f^TWg$ с $W=\mathrm{diag}(w_i)$, а
симметрия $L^T=L$ записывается как
$$WL=L^TW.$$
Она держится для любой сетки: это алгебра, а не точность квадратуры.


In [ ]:
from numpy.polynomial.legendre import leggauss

def sphere(polar=16, azimuthal=12):
    mu, wp = leggauss(polar)
    phi = 2*np.pi*np.arange(azimuthal)/azimuthal
    sin = np.sqrt(np.clip(1-mu*mu, 0., None))
    dirs = np.stack([np.outer(sin, np.cos(phi)).ravel(),
                     np.outer(sin, np.sin(phi)).ravel(),
                     np.repeat(mu, azimuthal)], axis=1)
    return dirs, np.repeat(wp*(2*np.pi/azimuthal), azimuthal), np.repeat(mu, azimuthal)

dirs, w, mu = sphere()
d0 = medium.extinction_per_m - 1j*OMEGA_PER_NS/medium.speed_m_per_ns
phase = hg_phase(dirs @ dirs.T, medium.g)
L = np.diag(d0 + 1j*K_PER_M*mu) - medium.scattering_per_m*phase*w[None, :]
W = np.diag(w)

print('направлений', len(w), ' телесный угол', w.sum(), ' 4pi =', 4*np.pi)
print('нормировка индикатриссы на сетке: max|1 - sum p w| =', np.max(np.abs(phase@w - 1)))
residual = np.max(np.abs(W@L - L.T@W))/np.max(np.abs(W@L))
print('относительная невязка WL = L^T W:', residual)
assert residual < 1e-14


## 2.4. Взаимность: один adjoint solve, много источников

Прямой путь решает $LI=q$ для каждого источника. Сопряжённый решает $Lh=a$
**один раз** и затем только сворачивает:
$$\widetilde R=\langle a,L^{-1}q\rangle_B=\langle h,q\rangle_B.$$
Дискретная дельта в узле $j$ даёт ровно $h_j$ — это и есть «один solve на все
направления источника».


In [ ]:
axis_det, axis_src = np.array([.6, 0., .8]), np.array([0., -.8, .6])
c = dirs @ axis_det
a = 0.40 + 0.48*c + 0.12*c*c            # приёмистость главы 3

j0 = int(np.argmax(dirs @ axis_src))
sources = {'изотропный': np.full(len(w), 1/(4*np.pi)),
           'узкая доля': hg_phase(dirs @ axis_src, 0.9),
           'дельта в узле': np.eye(len(w))[j0]/w[j0],
           'косинусная доля': np.clip(dirs @ axis_src, 0., None)}

h = np.linalg.solve(L, a)               # единственный сопряжённый solve
rows = []
for name, q in sources.items():
    direct = a @ (w*np.linalg.solve(L, q))
    adjoint = h @ (w*q)
    rows.append((name, direct, abs(direct-adjoint)/abs(direct)))
    assert rows[-1][2] < 1e-11
display(Markdown('| источник | прямой ответ | отн. расхождение |\n|---|---|---|\n'
                 + '\n'.join(f'| {n} | {d:.9g} | {e:.2e} |' for n, d, e in rows)))
print('дельта возвращает h(s_j):', abs(h @ (w*sources['дельта в узле']) - h[j0]))


## 2.5. Коэрцитивность

$L$ комплексно-симметричен, но **не** эрмитов. Для оценки обратимости нужна
другая пара — эрмитова. Стриминг чисто мнимый и в вещественную часть не даёт
ничего, нормированное рассеяние не превосходит единицы, поэтому
$$\mathrm{Re}\,(h,Lh)_H\ \ge\ \mu_a\|h\|^2.$$
Проверяем через $B=W^{1/2}LW^{-1/2}$ и его эрмитову часть. Дефицит границы
объясняется тем, насколько сетка нормирует индикатриссу.


In [ ]:
root = np.sqrt(w)
B = (root[:, None]*L)/root[None, :]
H = (B + B.conj().T)/2
eig = np.linalg.eigvalsh(H)
deficit = medium.absorption_per_m - eig[0]
bound = medium.scattering_per_m*np.max(np.abs(phase@w - 1))
print('lambda_min =', eig[0], ' mu_a =', medium.absorption_per_m)
print('lambda_max =', eig[-1], ' mu_t =', medium.extinction_per_m)
print('дефицит', deficit, ' <= mu_s * ошибка нормировки =', bound)
assert deficit <= bound + 1e-12

# Та же конструкция при mu_a = 0, k = omega = 0: изотропная нулевая мода.
s = medium.scattering_per_m
L0 = np.diag(np.full(len(w), s)) - s*phase*w[None, :]
print('L0 @ 1 (относительно mu_s):', np.max(np.abs(L0 @ np.ones(len(w))))/s)
print('следующее собственное значение ~ mu_s(1-g) =', s*(1-medium.g))


## Задания

1. Вывести $\partial_t\to-i\omega$ и $\nabla\to+i\mathbf k$ по частям и сказать,
какие граничные члены отброшены.
2. Показать, что $\langle f,Lg\rangle_B=\langle Lf,g\rangle_B$ требует только
симметрии индикатриссы, и найти в выводе место, где важно отсутствие сопряжения.
3. Построить $f$, для которой $(f,Lf)_H$ не вещественно при $\omega\neq0$, и
заключить, что комплексная симметрия не означает эрмитовость.
4. Измельчить сетку и проверить, что невязка $WL=L^TW$ не меняется, а дефицит
коэрцитивности падает вместе с ошибкой нормировки.
5. Заменить индикатриссу на несимметричную и показать численно, что взаимность
ломается, а прямой и сопряжённый ответы расходятся.

Код: `medium.py`, `single.py`, `angular.py`; книга: глава 4.
